<a href="https://colab.research.google.com/github/BRANOPODCAST/546sadfd/blob/main/claude_telfaza.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1 — Install (run once per session)
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "gradio>=4.0", "faster-whisper", "google-genai", "yt-dlp",
    "arabic-reshaper", "python-bidi", "Pillow", "opencv-python-headless",
    "edge-tts"], check=True)
subprocess.run(["apt-get", "install", "-y", "-q", "ffmpeg"], check=True)

CompletedProcess(args=['apt-get', 'install', '-y', '-q', 'ffmpeg'], returncode=0)

In [ ]:
#!/usr/bin/env python3
"""
╔══════════════════════════════════════════════════════════════════╗
║  TELFAZA ZMAN — Production UI                                   ║
║  Full Gradio interface: clip generation + dubbing               ║
║                                                                  ║
║  COLAB SETUP (run this cell first):                             ║
║    !pip install -q gradio faster-whisper google-genai yt-dlp    ║
║         arabic-reshaper python-bidi Pillow opencv-python-headless║
║    !apt install -y -q ffmpeg                                     ║
║    Then run this entire file as one cell, or:                   ║
║    exec(open('telfaza_ui.py').read())                           ║
╚══════════════════════════════════════════════════════════════════╝
"""
# ─── std lib ──────────────────────────────────────────────────
import os, sys, json, re, time, shutil, subprocess, tempfile, threading
from pathlib import Path
from datetime import datetime

# ─── third-party (installed at runtime if missing) ────────────
def _ensure(*pkgs):
    miss = []
    for p in pkgs:
        try: __import__(p.replace("-","_").split("[")[0])
        except ImportError: miss.append(p)
    if miss:
        subprocess.run([sys.executable,"-m","pip","install","-q",*miss], check=True)

_ensure("gradio>=4.0","faster_whisper","google-genai","yt_dlp",
        "arabic_reshaper","python-bidi","Pillow","opencv-python-headless")

# CUDA allocator tuning — reduces fragmentation-related OOM crashes that can
# silently kill the whole Colab kernel with no visible Python error
# (must be set before torch initializes CUDA)
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import gradio as gr
import torch
import gc
import cv2
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import arabic_reshaper
from bidi.algorithm import get_display

# ═══════════════════════════════════════════════════════════════
# CONSTANTS & HELPERS
# ═══════════════════════════════════════════════════════════════
WHISPER_MODELS   = ["large-v3","large-v2","medium","small","base"]
GEMINI_MODELS    = {
    "Gemini 2.5 Flash (fast, recommended)": "gemini-2.5-flash",
    "Gemini 2.5 Pro (best quality, slow)":  "gemini-2.5-pro",
    "Gemini 1.5 Flash (fallback)":           "gemini-1.5-flash",
    "Gemini 1.5 Pro (fallback)":             "gemini-1.5-pro",
}
LANGUAGES = {
    "🇲🇦 Moroccan Darija (Arabic)": "darija",
    "🇸🇦 Arabic (MSA)":             "ar",
    "🇬🇧 English":                  "en",
    "🇫🇷 French":                   "fr",
    "🇪🇸 Spanish":                  "es",
    "🔍 Auto-match source video":   "auto",
}
WHISPER_LANG = {
    "🔍 Auto-detect":  None,
    "🇲🇦 Arabic":      "ar",
    "🇬🇧 English":     "en",
    "🇫🇷 French":      "fr",
    "🇪🇸 Spanish":     "es",
    "🇩🇪 German":      "de",
    "🇮🇹 Italian":     "it",
}
WM_POSITIONS = ["bottom-center","top-center","top-right","top-left","bottom-right","bottom-left"]
WORK_DIR = Path("/content/telfaza_tmp")
FONTS_DIR = Path("/content/fonts")
DRIVE_DIR = Path("/content/drive/MyDrive/TelfazaZman/clips")
WORK_DIR.mkdir(parents=True, exist_ok=True)

_whisper_model_cache = {}
_log_lines = []

def log(msg, lines_ref=None):
    ts = datetime.now().strftime("%H:%M:%S")
    line = f"[{ts}] {msg}"
    print(line)
    if lines_ref is not None:
        lines_ref.append(line)

def run_ff(*args, label="ffmpeg"):
    cmd = ["ffmpeg", "-y"] + list(args)
    r = subprocess.run(cmd, capture_output=True)
    if r.returncode != 0:
        raise RuntimeError(f"FFmpeg {label}:\n{r.stderr.decode(errors='replace')[-400:]}")

def get_vid_info(path):
    def probe(key):
        r = subprocess.check_output(
            ["ffprobe","-v","error","-select_streams","v:0",
             "-show_entries",f"stream={key}","-of","csv=p=0",str(path)],
            text=True).strip()
        return r
    w, h = int(probe("width")), int(probe("height"))
    fps_s = probe("r_frame_rate"); num,den = fps_s.split("/")
    fps = float(num)/float(den)
    dur = float(subprocess.check_output(
        ["ffprobe","-v","error","-show_entries","format=duration",
         "-of","csv=p=0",str(path)], text=True).strip())
    return {"w":w,"h":h,"fps":fps,"duration":dur}

def ensure_amiri():
    fp = FONTS_DIR/"Amiri-Regular.ttf"
    if fp.exists(): return str(fp)
    FONTS_DIR.mkdir(exist_ok=True)
    subprocess.run(
        "curl -sL https://github.com/aliftype/amiri/releases/download/"
        "1.003/Amiri-1.003.zip -o /tmp/amiri.zip && "
        "unzip -qo /tmp/amiri.zip -d /tmp/amiri_x && "
        f"cp /tmp/amiri_x/Amiri-1.003/Amiri-Regular.ttf {fp}",
        shell=True, check=True)
    return str(fp)

# ═══════════════════════════════════════════════════════════════
# CORE PIPELINE FUNCTIONS
# ═══════════════════════════════════════════════════════════════

def download_video(url, out_dir):
    """Download YouTube/Facebook/any URL with fallback strategies."""
    out = str(out_dir / "source_video.mp4")
    import yt_dlp
    cookie_f = str(WORK_DIR/"yt_cookies.txt") if (WORK_DIR/"yt_cookies.txt").exists() else None
    base = {
        "format":"bestvideo[vcodec^=avc1][ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best",
        "merge_output_format":"mp4","outtmpl":out,"quiet":True,"overwrites":True,
        **({"cookiefile":cookie_f} if cookie_f else {})
    }
    for client in [["tv_embed","android","mweb"],["mweb","web"],[None]]:
        try:
            opts = {**base}
            if client[0]:
                opts["extractor_args"] = {"youtube":{"player_client":client}}
            with yt_dlp.YoutubeDL(opts) as ydl:
                ydl.download([url])
            if os.path.exists(out):
                return out
        except Exception as e:
            last_e = e
    raise RuntimeError(f"Download failed: {last_e}")

def transcribe(video_path, model_name, lang_code, logs):
    """Whisper transcription with GPU auto-detect."""
    global _whisper_model_cache
    from faster_whisper import WhisperModel
    cache_key = (model_name, lang_code)
    if cache_key not in _whisper_model_cache:
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        cmp = "float16" if dev == "cuda" else "int8"
        log(f"Loading Whisper {model_name} on {dev.upper()} ({cmp})…", logs)
        cache_dir = "/content/drive/MyDrive/TelfazaZman/whisper_models" if Path("/content/drive").exists() else str(WORK_DIR/"whisper_models")
        Path(cache_dir).mkdir(parents=True, exist_ok=True)
        _whisper_model_cache[cache_key] = WhisperModel(
            model_name, device=dev, compute_type=cmp, download_root=cache_dir)
    model = _whisper_model_cache[cache_key]
    kw = dict(word_timestamps=True, beam_size=5, vad_filter=True,
               vad_parameters=dict(min_silence_duration_ms=400))
    if lang_code:
        kw["language"] = lang_code
    log("Transcribing…", logs)
    segs, info = model.transcribe(str(video_path), **kw)
    tr = {"segments":[], "language":info.language, "text":""}
    for s in segs:
        words = [{"word":w.word,"start":w.start,"end":w.end} for w in (s.words or [])]
        tr["segments"].append({"start":s.start,"end":s.end,"text":s.text.strip(),"words":words})
        tr["text"] += s.text + " "
    tr["text"] = tr["text"].strip()
    log(f"Detected language: {info.language} (prob={info.language_probability:.2f})", logs)
    return tr

def call_gemini(client, model_id, prompt, fallback_ids, logs):
    """Call Gemini with automatic model fallback on 503."""
    models_to_try = [model_id] + [m for m in fallback_ids if m != model_id]
    for mid in models_to_try:
        for attempt in range(3):
            try:
                r = client.models.generate_content(model=mid, contents=prompt)
                return r.text, mid
            except Exception as e:
                if "503" in str(e) or "UNAVAILABLE" in str(e) or "quota" in str(e).lower():
                    wait = 20*(attempt+1)
                    log(f"⏳ Gemini {mid} unavailable — retry in {wait}s (attempt {attempt+1}/3)", logs)
                    time.sleep(wait)
                elif "RESOURCE_EXHAUSTED" in str(e):
                    log(f"⚠️ Rate limit on {mid}, trying next model…", logs)
                    break
                else:
                    raise
        log(f"Switching from {mid} to next fallback…", logs)
    raise RuntimeError("All Gemini models failed")

def detect_clips(transcript, vid_dur, gemini_key, model_id, fallback_ids,
                  output_lang, min_dur, max_clips, logs):
    """Use Gemini to find viral clips."""
    from google import genai
    client = genai.Client(api_key=gemini_key)
    words = [{"w":w["word"],"s":w["start"],"e":w["end"]}
             for s in transcript["segments"] for w in s.get("words",[])]

    lang_rule = {
        "darija": "Moroccan Darija Arabic (الدارجة المغربية)",
        "ar":     "Modern Standard Arabic (الفصحى)",
        "en":     "English",
        "fr":     "French",
        "es":     "Spanish",
        "auto":   f"the same language as the video ({transcript['language']})",
    }.get(output_lang, "Arabic")

    prompt = f"""You are a top viral short-form video editor.

LANGUAGE RULE: ALL text output fields MUST be in {lang_rule}.

VIDEO_DURATION: {vid_dur:.1f}s | SOURCE_LANG: {transcript['language']}

Find {min(max_clips,12)} to {max_clips} MOST VIRAL clips (minimum {max_clips//2}).
CLIP DURATION RULE: minimum 60 seconds, target 60-90 seconds. NEVER shorter than 55s.

TRANSCRIPT: {transcript['text'][:3000]}

WORD_TIMESTAMPS: {json.dumps(words[:1500])}

Return ONLY valid JSON:
{{"shorts":[{{"start":12.3,"end":75.6,
  "viral_hook_text":"hook in {lang_rule}",
  "title":"title",
  "description_for_facebook":"caption + hashtags",
  "tags":["#tag1"],
  "speakers":1}}]}}"""

    log(f"Asking Gemini for clips (output lang: {output_lang})…", logs)
    raw, used_model = call_gemini(client, model_id, prompt, fallback_ids, logs)
    log(f"✅ Gemini responded using {used_model}", logs)
    raw = raw.strip().lstrip("```json").rstrip("```").strip()
    data = json.loads(raw)
    clips_raw = data.get("shorts", data.get("clips", []))
    clips = [c for c in clips_raw
             if (float(c.get("end",0)) - float(c.get("start",0))) >= 55]
    log(f"Found {len(clips_raw)} clips → {len(clips)} pass 55s filter", logs)
    return clips, client

def make_srt(transcript, cs, ce, out):
    words = [w for s in transcript["segments"]
             for w in s.get("words",[])
             if w["end"]>cs and w["start"]<ce]
    if not words: return False
    def ft(x):
        x=max(0.,float(x));h=int(x//3600);m=int((x%3600)//60)
        return f"{h:02d}:{m:02d}:{int(x%60):02d},{int((x-int(x))*1000):03d}"
    srt,idx,blk,t0="",1,[],None
    for w in words:
        ts=max(0.,w["start"]-cs)
        if not blk: blk,t0=[w],ts
        else:
            if sum(len(x["word"])+1 for x in blk)+len(w["word"])>28 or ts-t0>2.5:
                te=max(0.,blk[-1]["end"]-cs)
                srt+=f"{idx}\n{ft(t0)} --> {ft(te)}\n{' '.join(x['word'].strip() for x in blk)}\n\n"
                idx+=1; blk,t0=[w],ts
            else: blk.append(w)
    if blk:
        te=max(0.,blk[-1]["end"]-cs)
        srt+=f"{idx}\n{ft(t0)} --> {ft(te)}\n{' '.join(x['word'].strip() for x in blk)}\n\n"
    Path(out).write_text(srt,encoding="utf-8"); return True

def make_watermark(text, out, opacity=0.8):
    fp = ensure_amiri()
    try: font=ImageFont.truetype(fp,34)
    except: font=ImageFont.load_default()
    txt = get_display(arabic_reshaper.reshape(text))
    d=ImageDraw.Draw(Image.new("RGBA",(1,1)))
    bb=d.textbbox((0,0),txt,font=font)
    w,h=bb[2]-bb[0]+24,bb[3]-bb[1]+14
    img=Image.new("RGBA",(w,h),(0,0,0,0))
    draw=ImageDraw.Draw(img)
    draw.text((2,2),txt,font=font,fill=(0,0,0,int(180*opacity)))
    draw.text((0,0),txt,font=font,fill=(255,255,255,int(255*opacity)))
    img.save(out); return w,h

def detect_faces(path, start, end, n=8):
    """OpenCV Haar cascade face detection (no mediapipe)."""
    fc=cv2.CascadeClassifier(cv2.data.haarcascades+'haarcascade_frontalface_default.xml')
    cap=cv2.VideoCapture(str(path))
    dur=max(end-start,0.1)
    times=[start+dur*i/max(n-1,1) for i in range(n)]
    all_faces=[]
    for t in times:
        cap.set(cv2.CAP_PROP_POS_MSEC,t*1000)
        ret,frame=cap.read()
        if not ret: continue
        h,w=frame.shape[:2]
        gray=cv2.equalizeHist(cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY))
        dets=fc.detectMultiScale(gray,1.1,4,minSize=(max(30,w//20),max(30,h//20)))
        ff=[{"cx":float(x+fw/2),"w":float(fw),"h":float(fh)} for x,y,fw,fh in dets]
        ff.sort(key=lambda f:f["cx"])
        all_faces.append(ff)
    cap.release(); return all_faces

def decide_layout(all_faces, split_thr=0.6):
    n=len(all_faces)
    if n==0: return "blur_bg"
    multi=sum(1 for f in all_faces if len(f)>=2)
    single=sum(1 for f in all_faces if len(f)==1)
    if multi/n>=split_thr: return "split"
    if single/n>=0.35: return "single"
    return "blur_bg"

def build_clip(src, start, end, idx, transcript,
               wm_text, wm_pos, wm_opacity, wm_logo,
               neon_color, sub_font_size, logs):
    slug=f"clip_{idx:02d}"
    raw=str(WORK_DIR/f"{slug}_raw.mp4")
    vert=str(WORK_DIR/f"{slug}_vert.mp4")
    sub=str(WORK_DIR/f"{slug}_sub.mp4")
    final=str(WORK_DIR/f"{slug}_final.mp4")
    srt_p=str(WORK_DIR/f"{slug}.srt")
    wm_p=str(WORK_DIR/"watermark.png")
    OW,OH=1080,1920

    run_ff("-ss",str(start),"-to",str(end),"-i",str(src),
           "-c:v","libx264","-crf","18","-preset","fast","-c:a","aac",raw,label="cut")

    info=get_vid_info(raw)
    vw,vh=info["w"],info["h"]
    afaces=detect_faces(raw,0,info["duration"])
    layout=decide_layout(afaces)
    log(f"  Layout: {layout}",logs)

    if layout=="single":
        cxs=[f["cx"] for ff in afaces for f in ff]
        cx=int(np.mean(cxs)) if cxs else vw//2
        cw=min(int(vh*9/16),vw)
        cx=max(cw//2,min(cx,vw-cw//2))
        vf=f"[0:v]crop={cw}:{vh}:{cx-cw//2}:0,scale={OW}:{OH}[out]"
    elif layout=="split":
        left=[f["cx"] for ff in afaces if len(ff)>=2 for f in [sorted(ff,key=lambda x:x["cx"])[0]]]
        right=[f["cx"] for ff in afaces if len(ff)>=2 for f in [sorted(ff,key=lambda x:x["cx"])[-1]]]
        cx1=int(np.mean(left)) if left else vw//3
        cx2=int(np.mean(right)) if right else 2*vw//3
        hh=OH//2; cw=min(int(vh*9/16),vw)
        x1=max(0,min(int(cx1-cw//2),vw-cw))
        x2=max(0,min(int(cx2-cw//2),vw-cw))
        ny=hh-4
        vf=(f"[0:v]crop={cw}:{vh}:{x1}:0,scale={OW}:{hh}[top];"
            f"[0:v]crop={cw}:{vh}:{x2}:0,scale={OW}:{hh}[bot];"
            f"[top][bot]vstack[st];"
            f"[st]drawbox=x=0:y={ny-6}:w={OW}:h=20:color={neon_color}@0.25:t=fill,"
            f"drawbox=x=0:y={ny}:w={OW}:h=8:color={neon_color}@0.9:t=fill[out]")
    else:
        vf=(f"[0:v]scale={OW}:{OH}:force_original_aspect_ratio=increase,"
            f"crop={OW}:{OH},gblur=sigma=28[bg];"
            f"[0:v]scale={OW}:-2[fg];"
            f"[bg][fg]overlay=0:(H-h)/2[out]")

    run_ff("-i",raw,"-filter_complex",vf,
           "-map","[out]","-map","0:a?",
           "-c:v","libx264","-preset","fast","-crf","20","-c:a","aac",vert,label="layout")

    # Subtitles
    cur=vert
    if make_srt(transcript, start, end, srt_p):
        safe=srt_p.replace(":","\\:")
        sty=(f"Alignment=2,Fontname=Amiri,Fontsize={sub_font_size},"
             f"PrimaryColour=&H00FFFFFF,OutlineColour=&H00000000,"
             f"BorderStyle=1,Outline=2,Shadow=0,Bold=1,MarginV=35")
        try:
            run_ff("-i",vert,"-vf",
                   f"subtitles='{safe}':fontsdir='/content/fonts':force_style='{sty}'",
                   "-c:a","copy","-c:v","libx264","-preset","fast","-crf","20",sub,label="subs")
            cur=sub
        except Exception as e:
            log(f"  ⚠️ Subtitle failed (skipping): {e}",logs)

    # Watermark
    wm_src=wm_logo if wm_logo and os.path.exists(wm_logo) else None
    if not wm_src and wm_text:
        ww,wh=make_watermark(wm_text,wm_p,wm_opacity)
        wm_src=wm_p
    if wm_src:
        pos_map={"bottom-center":f"(W-w)/2:H-h-20","top-center":f"(W-w)/2:20",
                 "top-right":"W-w-20:20","top-left":"20:20",
                 "bottom-right":"W-w-20:H-h-20","bottom-left":"20:H-h-20"}
        xy=pos_map.get(wm_pos,"(W-w)/2:H-h-20")
        run_ff("-i",cur,"-i",wm_src,
               "-filter_complex",f"[0:v][1:v]overlay={xy}[out]",
               "-map","[out]","-map","0:a?",
               "-c:v","libx264","-preset","fast","-crf","20",
               "-c:a","aac","-movflags","+faststart",final,label="watermark")
    else:
        shutil.copy(cur, final)
    return final, layout

def gen_facebook_meta(clip, clip_text, output_lang, gemini_key, model_id, fallback_ids, logs):
    from google import genai
    client=genai.Client(api_key=gemini_key)
    lang_name={"darija":"الدارجة المغربية","ar":"العربية الفصحى",
               "en":"English","fr":"Français","es":"Español"}.get(output_lang,"الدارجة")
    prompt=f"""Generate Facebook Reels metadata in {lang_name}.
Hook: {clip.get('viral_hook_text','')}
Content: {clip_text[:300]}
Return JSON only:
{{"title":"<80 chars>","caption":"150-300 chars + hashtags","hashtags":["#tag"],"hook_first":"<50 chars first line"}}"""
    try:
        raw,_=call_gemini(client, model_id, prompt, fallback_ids, logs)
        raw=raw.strip().lstrip("```json").rstrip("```").strip()
        return json.loads(raw)
    except:
        return {"title":clip.get("viral_hook_text",""),"caption":"","hashtags":[],"hook_first":""}

# ═══════════════════════════════════════════════════════════════
# GPU MEMORY MANAGEMENT — prevents the silent CUDA-OOM crashes that
# force a full Colab runtime restart with no visible Python error
# ═══════════════════════════════════════════════════════════════
def gpu_mem_str():
    if not torch.cuda.is_available():
        return "CPU mode"
    free, total = torch.cuda.mem_get_info()
    return f"{(total-free)/1e9:.1f}/{total/1e9:.1f} GB VRAM used"

def free_gpu(logs=None, clear_whisper=False):
    """Release cached models + CUDA cache. Call before loading a new heavy
    model (Whisper → TTS) so two large models never compete for VRAM at once."""
    global _whisper_model_cache
    if clear_whisper and _whisper_model_cache:
        for k in list(_whisper_model_cache.keys()):
            del _whisper_model_cache[k]
        _whisper_model_cache = {}
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    if logs is not None:
        log(f"🧹 Memory freed — {gpu_mem_str()}", logs)

def free_gpu_ui():
    """'Free GPU Memory' button handler — recovers stuck models without a runtime restart."""
    l = []
    free_gpu(l, clear_whisper=True)
    return "\n".join(l) if l else f"🧹 Freed — {gpu_mem_str()}"

# ═══════════════════════════════════════════════════════════════
# DUBBING QUALITY HELPERS — clean reference audio, sentence
# grouping, gentle time-stretching. These three are what turn
# "robotic, not cloned" output into natural cloned speech.
# ═══════════════════════════════════════════════════════════════
_SENTENCE_END = re.compile(r'[.!?؟。]\s*$')

def extract_best_reference_audio(video_path, transcript, out_path, target_dur=(6, 15)):
    """
    Pick the longest single Whisper segment within target_dur seconds of
    continuous speech as the voice-cloning reference. Blindly grabbing the
    first N seconds often captures intro music or silence — a bad reference
    is the #1 cause of "doesn't sound cloned at all".
    """
    segs = transcript.get("segments", [])
    candidates = [s for s in segs
                   if target_dur[0] <= (s["end"]-s["start"]) <= target_dur[1]
                   and len(s["text"].strip()) > 8]
    if candidates:
        best = max(candidates, key=lambda s: s["end"]-s["start"])
        start, dur = best["start"], best["end"]-best["start"]
    else:
        start, dur = 0.0, 10.0
        for s in segs:
            if s["end"] >= target_dur[0]:
                dur = min(max(s["end"], target_dur[0]), target_dur[1])
                break
    run_ff("-i", str(video_path), "-ss", str(start), "-t", str(dur),
           "-vn", "-acodec","pcm_s16le","-ar","22050","-ac","1",
           out_path, label="extract reference audio")
    return start, dur

def group_into_sentences(transcript, max_chars=220, max_gap=1.2):
    """
    Merge Whisper's word-level fragments into sentence-sized chunks.
    Whisper often emits dozens of 1-3 word segments (e.g. '¿Qué?' x15).
    Running TTS on each tiny fragment separately destroys prosody and
    sounds robotic when concatenated — TTS on full sentences sounds natural.
    """
    segs = transcript["segments"]
    groups, cur_text, cur_start, cur_end = [], "", None, None
    for s in segs:
        txt = s["text"].strip()
        if not txt:
            continue
        if cur_start is None:
            cur_start, cur_end, cur_text = s["start"], s["end"], txt
            continue
        gap = s["start"] - cur_end
        if gap > max_gap or len(cur_text)+len(txt) > max_chars or _SENTENCE_END.search(cur_text):
            groups.append({"start":cur_start,"end":cur_end,"text":cur_text.strip()})
            cur_start, cur_end, cur_text = s["start"], s["end"], txt
        else:
            cur_text += " " + txt
            cur_end = s["end"]
    if cur_text:
        groups.append({"start":cur_start,"end":cur_end,"text":cur_text.strip()})
    return groups

def time_stretch_audio(in_path, out_path, speed):
    """
    Stretch audio to `speed` ratio (gen_duration / target_duration).
    Uses rubberband (preserves pitch & formants) when ffmpeg supports it,
    falls back to atempo. Clamped to 0.85-1.15 — wider ratios with atempo
    sound like a chipmunk/robot, so beyond this range we accept mild drift
    instead of destroying naturalness.
    """
    speed = max(0.85, min(speed, 1.15))
    if abs(speed-1.0) < 0.02:
        shutil.copy(in_path, out_path)
        return
    try:
        run_ff("-i", in_path, "-af", f"rubberband=tempo={speed}", out_path, label="rubberband")
    except Exception:
        run_ff("-i", in_path, "-filter:a", f"atempo={speed}", out_path, label="atempo")

# ═══════════════════════════════════════════════════════════════
# DUBBING PIPELINE
# ═══════════════════════════════════════════════════════════════

def dub_video(video_path, transcript, target_lang_code, model_choice,
              gemini_key, model_id, fallback_ids, logs):
    """
    Dub a video to target_lang_code while preserving the original speaker's
    voice. Pipeline: free GPU → extract a clean reference clip → group the
    transcript into sentences → translate → clone voice per sentence →
    gentle time-align → mix with original audio.
    """
    from google import genai
    client = genai.Client(api_key=gemini_key)

    out_dir = WORK_DIR / "dubbed"
    out_dir.mkdir(exist_ok=True)
    vid_name = Path(video_path).stem
    final_out = str(out_dir / f"{vid_name}_dubbed_{target_lang_code}.mp4")

    # Step 0: Whisper is no longer needed — free its VRAM before loading TTS
    log(f"GPU before cleanup: {gpu_mem_str()}", logs)
    free_gpu(logs, clear_whisper=True)

    # Step 1: Extract a clean reference clip (longest continuous sentence, 6-15s)
    ref_audio = str(out_dir / "ref_voice.wav")
    rstart, rdur = extract_best_reference_audio(video_path, transcript, ref_audio)
    log(f"✅ Voice reference: {rdur:.1f}s starting at {rstart:.1f}s", logs)

    # Step 2: Group raw Whisper fragments into full sentences
    groups = group_into_sentences(transcript)
    log(f"Grouped {len(transcript['segments'])} fragments → {len(groups)} sentences", logs)

    # Step 3: Translate sentence-by-sentence with Gemini
    texts = [g["text"] for g in groups]
    src_lang = transcript["language"]
    lang_name = {"ar":"Arabic","en":"English","fr":"French","es":"Spanish",
                 "darija":"Moroccan Darija Arabic","de":"German","it":"Italian",
                 "zh":"Chinese"}.get(target_lang_code, target_lang_code)

    log(f"Translating {len(groups)} sentences to {lang_name}…", logs)
    texts_sent = texts[:80]
    if len(texts) > 80:
        log(f"⚠️ {len(texts)} sentences found — translating first 80", logs)
        groups = groups[:80]
    t_prompt = (f"Translate these {src_lang} sentences to {lang_name}. "
                f"Keep each translation natural and roughly the same length/pacing "
                f"as the original so it can be dubbed over the same timing. "
                f"Return ONLY a JSON array of strings, same order, same count:\n"
                f"{json.dumps(texts_sent)}")
    raw, _ = call_gemini(client, model_id, t_prompt, fallback_ids, logs)
    raw = raw.strip().lstrip("```json").rstrip("```").strip()
    try:
        translations = json.loads(raw)
        if len(translations) != len(texts_sent):
            raise ValueError(f"got {len(translations)} translations for {len(texts_sent)} sentences")
    except Exception as e:
        log(f"⚠️ Translation parse issue ({e}) — using original text untranslated", logs)
        translations = texts_sent
    if target_lang_code != src_lang and translations == texts_sent:
        log("⚠️⚠️ Output is UNTRANSLATED (same as source text) — translation step failed silently", logs)
    log(f"✅ {len(translations)} sentences translated", logs)

    # Step 4: TTS with voice cloning
    free_gpu(logs)
    if model_choice.startswith("Chatterbox Multilingual"):
        dubbed_audio, used = _dub_chatterbox_multilingual(groups, translations, ref_audio, out_dir, target_lang_code, logs)
    elif model_choice.startswith("XTTS-v2"):
        dubbed_audio, used = _dub_xtts(groups, translations, ref_audio, out_dir, target_lang_code, logs)
    elif model_choice.startswith("Chatterbox (English"):
        dubbed_audio, used = _dub_chatterbox(groups, translations, ref_audio, out_dir, logs)
    else:
        dubbed_audio, used = _dub_edge_tts(groups, translations, out_dir, target_lang_code, logs), "edge-tts"

    if not dubbed_audio or not os.path.exists(dubbed_audio):
        raise RuntimeError("TTS generation produced no audio")

    if used == "edge-tts":
        log("⚠️⚠️⚠️ FALLBACK: Edge TTS used — generic voice, NOT cloned from the original speaker ⚠️⚠️⚠️", logs)

    # Step 5: Mix dubbed audio with video (original kept very low for ambience)
    run_ff("-i", video_path, "-i", dubbed_audio,
           "-filter_complex",
           "[0:a]aformat=sample_rates=44100:channel_layouts=stereo,volume=0.08[orig];"
           "[1:a]aformat=sample_rates=44100:channel_layouts=stereo,volume=1.0[dub];"
           "[orig][dub]amix=inputs=2:duration=shortest:normalize=0[a]",
           "-map","0:v","-map","[a]",
           "-c:v","copy","-c:a","aac","-shortest", final_out,
           label="mix dubbed audio")
    log(f"✅ Dubbed video ({used}): {final_out}", logs)
    free_gpu(logs)
    return final_out, used

def _dub_chatterbox_multilingual(groups, translations, ref_audio, out_dir, lang_code, logs):
    """
    Chatterbox Multilingual (Resemble AI, MIT license) — 23 languages incl.
    Arabic, zero-shot voice cloning from ~10s reference, cross-language voice
    transfer (e.g. clone a Spanish voice → speak Arabic in that voice).
    Recommended default for Telfaza Zman.
    Note: output carries Resemble's inaudible PerTh watermark (standard for
    this model — survives re-encoding, identifies the audio as AI-generated).
    """
    log("Loading Chatterbox Multilingual (first run downloads ~1-2GB)…", logs)
    try:
        _ensure("chatterbox-tts")
        import torchaudio as ta
        from chatterbox.mtl_tts import ChatterboxMultilingualTTS
        device = "cuda" if torch.cuda.is_available() else "cpu"
        model = ChatterboxMultilingualTTS.from_pretrained(device=device)
        log(f"  Loaded on {device.upper()} — {gpu_mem_str()}", logs)
    except Exception as e:
        log(f"⚠️ Chatterbox Multilingual load failed ({e}) — falling back to XTTS-v2", logs)
        return _dub_xtts(groups, translations, ref_audio, out_dir, lang_code, logs)

    cb_lang = {"ar":"ar","darija":"ar","en":"en","fr":"fr","es":"es",
               "de":"de","it":"it","zh":"zh"}.get(lang_code,"en")

    audio_files = []
    for i,(g,text) in enumerate(zip(groups, translations)):
        if not text.strip(): continue
        try:
            try:
                # cfg_weight=0 when reference language != target language,
                # per Resemble's docs — avoids the clone inheriting the
                # reference clip's accent during cross-language transfer
                wav = model.generate(text, language_id=cb_lang,
                                     audio_prompt_path=ref_audio,
                                     exaggeration=0.5, cfg_weight=0.0)
            except TypeError:
                wav = model.generate(text, language_id=cb_lang, audio_prompt_path=ref_audio)
            seg_out = str(out_dir/f"seg_{i:04d}.wav")
            ta.save(seg_out, wav, model.sr)

            target_dur = g["end"] - g["start"]
            gen_dur = wav.shape[-1] / model.sr
            if target_dur > 0.3 and gen_dur > 0.3:
                stretched = seg_out.replace(".wav","_s.wav")
                time_stretch_audio(seg_out, stretched, gen_dur/target_dur)
                seg_out = stretched
            audio_files.append((g["start"], seg_out))
        except Exception as e:
            log(f"  ⚠️ Sentence {i+1}/{len(groups)} failed: {e}", logs)

    free_gpu(logs)
    out = _assemble_audio(audio_files, groups, out_dir)
    if not out:
        log("⚠️ Chatterbox Multilingual produced nothing — falling back to XTTS-v2", logs)
        return _dub_xtts(groups, translations, ref_audio, out_dir, lang_code, logs)
    return out, "chatterbox-multilingual"

def _dub_xtts(groups, translations, ref_audio, out_dir, lang_code, logs):
    """XTTS-v2 voice cloning — 17 languages incl. Arabic, ~6s reference."""
    log("Loading XTTS-v2 (first run downloads ~2GB)…", logs)
    try:
        _ensure("TTS")
        from TTS.api import TTS
        tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2")
        if torch.cuda.is_available():
            tts = tts.to("cuda")
        log(f"  Loaded — {gpu_mem_str()}", logs)
    except Exception as e:
        log(f"⚠️ XTTS-v2 load failed ({e}) — falling back to Edge TTS", logs)
        return _dub_edge_tts(groups, translations, out_dir, lang_code, logs), "edge-tts"

    xtts_lang = {"ar":"ar","en":"en","fr":"fr","es":"es",
                 "darija":"ar","de":"de","it":"it","zh":"zh-cn"}.get(lang_code,"en")

    audio_files = []
    for i,(g,text) in enumerate(zip(groups, translations)):
        if not text.strip(): continue
        seg_out = str(out_dir/f"seg_{i:04d}.wav")
        try:
            tts.tts_to_file(text=text, speaker_wav=ref_audio,
                           language=xtts_lang, file_path=seg_out)
            gen_dur = float(subprocess.check_output(
                ["ffprobe","-v","error","-show_entries","format=duration",
                 "-of","csv=p=0",seg_out], text=True).strip())
            target_dur = g["end"] - g["start"]
            if target_dur > 0.3 and gen_dur > 0.3:
                stretched = seg_out.replace(".wav","_s.wav")
                time_stretch_audio(seg_out, stretched, gen_dur/target_dur)
                seg_out = stretched
            audio_files.append((g["start"], seg_out))
        except Exception as e:
            log(f"  ⚠️ Sentence {i+1}/{len(groups)} failed: {e}", logs)

    free_gpu(logs)
    out = _assemble_audio(audio_files, groups, out_dir)
    if not out:
        log("⚠️ XTTS-v2 produced nothing — falling back to Edge TTS", logs)
        return _dub_edge_tts(groups, translations, out_dir, lang_code, logs), "edge-tts"
    return out, "xtts-v2"

def _dub_edge_tts(segs, translations, out_dir, lang_code, logs):
    """Free TTS via Edge TTS (no GPU, no voice clone, but works everywhere)."""
    log("Using Edge TTS (free, no voice cloning)…", logs)
    _ensure("edge-tts")
    import asyncio, edge_tts
    voice_map = {
        "ar":"ar-MA-JamalNeural","darija":"ar-MA-JamalNeural",
        "en":"en-US-GuyNeural","fr":"fr-FR-HenriNeural",
        "es":"es-ES-AlvaroNeural","de":"de-DE-ConradNeural",
    }
    voice = voice_map.get(lang_code,"en-US-GuyNeural")
    audio_files = []
    for i,(seg,text) in enumerate(zip(segs, translations)):
        if not text.strip(): continue
        out = str(out_dir/f"seg_{i:04d}.mp3")
        async def gen(t=text, o=out):
            c = edge_tts.Communicate(t, voice)
            await c.save(o)
        asyncio.run(gen())
        audio_files.append((seg["start"], out))
    return _assemble_audio(audio_files, segs, out_dir)

def _dub_chatterbox(groups, translations, ref_audio, out_dir, logs):
    """Chatterbox (English) — best English-only quality, zero-shot cloning."""
    log("Loading Chatterbox (English)…", logs)
    try:
        _ensure("chatterbox-tts")
        import torchaudio as ta
        from chatterbox.tts import ChatterboxTTS
        device = "cuda" if torch.cuda.is_available() else "cpu"
        model = ChatterboxTTS.from_pretrained(device=device)
        log(f"  Loaded — {gpu_mem_str()}", logs)
    except Exception as e:
        log(f"⚠️ Chatterbox failed ({e}) — falling back to Edge TTS", logs)
        return _dub_edge_tts(groups, translations, out_dir, "en", logs), "edge-tts"

    audio_files = []
    for i,(g,text) in enumerate(zip(groups, translations)):
        if not text.strip(): continue
        try:
            wav = model.generate(text, audio_prompt_path=ref_audio)
            seg_out = str(out_dir/f"seg_{i:04d}.wav")
            ta.save(seg_out, wav, model.sr)
            target_dur = g["end"]-g["start"]
            gen_dur = wav.shape[-1]/model.sr
            if target_dur > 0.3 and gen_dur > 0.3:
                stretched = seg_out.replace(".wav","_s.wav")
                time_stretch_audio(seg_out, stretched, gen_dur/target_dur)
                seg_out = stretched
            audio_files.append((g["start"], seg_out))
        except Exception as e:
            log(f"  ⚠️ Sentence {i+1}/{len(groups)} failed: {e}", logs)

    free_gpu(logs)
    out = _assemble_audio(audio_files, groups, out_dir)
    if not out:
        log("⚠️ Chatterbox produced nothing — falling back to Edge TTS", logs)
        return _dub_edge_tts(groups, translations, out_dir, "en", logs), "edge-tts"
    return out, "chatterbox-en"

def _assemble_audio(audio_files, segs, out_dir):
    """Assemble segments into a single audio track with correct timing."""
    if not audio_files: return None
    total_dur = segs[-1]["end"] + 2 if segs else 60
    # Create silent base
    silence = str(out_dir/"silence_base.wav")
    run_ff("-f","lavfi","-i",f"anullsrc=r=22050:cl=mono",
           "-t",str(total_dur),silence)
    # Build complex filter
    inputs = ["-i", silence]
    filters, delays = [], []
    for j,(ts, fp) in enumerate(audio_files):
        inputs += ["-i", fp]
        delay_ms = int(ts * 1000)
        filters.append(f"[{j+1}:a]adelay={delay_ms}|{delay_ms}[d{j}]")
        delays.append(f"[d{j}]")
    if not filters:
        return None
    mix = f"{''.join(delays)}amix=inputs={len(delays)}:normalize=0[mix]"
    full_filter = ";".join(filters) + ";" + mix
    out_mix = str(out_dir/"full_dubbed.wav")
    run_ff(*inputs, "-filter_complex", full_filter,
           "-map","[mix]","-t",str(total_dur), out_mix)
    return out_mix

# ═══════════════════════════════════════════════════════════════
# GRADIO UI
# ═══════════════════════════════════════════════════════════════

def run_pipeline(video_input, url_input, video_source,
                 gemini_key, primary_model, fallback_model,
                 whisper_model, whisper_lang_choice,
                 output_lang_choice,
                 min_clip_dur, max_clips,
                 wm_text, wm_pos, wm_opacity, wm_logo,
                 neon_color, sub_size,
                 cookies_file,
                 progress=gr.Progress()):
    """Main pipeline — called by Gradio Generate button."""
    logs = []
    results = []
    meta_outputs = []
    status = ""

    try:
        # 0. Setup
        WORK_DIR.mkdir(exist_ok=True)
        log(f"GPU: {gpu_mem_str()}", logs)
        gemini_key = gemini_key.strip()
        if not gemini_key:
            yield "❌ Gemini API key required", [], "No key provided"
            return

        model_id = GEMINI_MODELS.get(primary_model, "gemini-2.5-flash")
        fallback_ids = [GEMINI_MODELS.get(fallback_model,"gemini-1.5-flash"),
                        "gemini-1.5-flash","gemini-1.5-pro"]
        output_lang = LANGUAGES.get(output_lang_choice,"darija")
        lang_code = WHISPER_LANG.get(whisper_lang_choice, None)

        # Handle cookies
        if cookies_file is not None:
            shutil.copy(cookies_file, str(WORK_DIR/"yt_cookies.txt"))
            log("🍪 YouTube cookies loaded", logs)

        # 1. Video acquisition
        progress(0.05, desc="Getting video…")
        if video_source == "Upload file" and video_input:
            src = video_input
            log(f"Using uploaded file: {src}", logs)
        elif video_source == "URL (YouTube/Facebook)" and url_input.strip():
            log(f"Downloading: {url_input}", logs)
            src = download_video(url_input.strip(), WORK_DIR)
            log(f"✅ Downloaded: {src}", logs)
        else:
            yield "❌ Provide a video file or URL", [], "No input"
            return

        # 2. Transcribe
        progress(0.15, desc="Transcribing audio…")
        transcript = transcribe(src, whisper_model, lang_code, logs)
        detected = transcript["language"]
        log(f"Language detected: {detected}", logs)
        yield "\n".join(logs[-6:]), [], f"Transcribed. Detected: {detected}"

        # 3. Detect viral clips
        progress(0.30, desc="Finding viral moments…")
        vid_info = get_vid_info(src)
        clips, _ = detect_clips(transcript, vid_info["duration"],
                                  gemini_key, model_id, fallback_ids,
                                  output_lang, min_clip_dur, int(max_clips), logs)
        log(f"✅ {len(clips)} clips to process", logs)
        yield "\n".join(logs[-6:]), [], f"{len(clips)} clips found"

        # 4. Process each clip
        out_paths = []
        for i, clip in enumerate(clips):
            s,e = float(clip["start"]), float(clip["end"])
            dur = e - s
            progress((0.35 + 0.55*i/len(clips)), desc=f"Clip {i+1}/{len(clips)}…")
            log(f"\n🎬 Clip {i+1}: [{s:.1f}s→{e:.1f}s] ({dur:.0f}s)", logs)

            try:
                final, layout = build_clip(
                    src, s, e, i+1, transcript,
                    wm_text, wm_pos, float(wm_opacity)/100,
                    wm_logo, neon_color, int(sub_size), logs)

                # Save to Drive if mounted
                if DRIVE_DIR.parent.parent.exists():
                    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
                    dest = DRIVE_DIR/f"clip_{i+1:02d}_{datetime.now().strftime('%H%M%S')}.mp4"
                    shutil.copy(final, dest)
                    log(f"  💾 Saved to Drive: {dest.name}", logs)

                out_paths.append(final)
                log(f"  ✅ Done ({layout})", logs)

            except Exception as ex:
                log(f"  ❌ Failed: {ex}", logs)

            yield "\n".join(logs[-8:]), out_paths, f"Clip {i+1}/{len(clips)} done"

        # 5. Generate Facebook metadata
        progress(0.95, desc="Generating Facebook metadata…")
        meta_lines = []
        for i,(clip,path) in enumerate(zip(clips, out_paths)):
            seg_text=" ".join(s["text"] for s in transcript["segments"]
                             if clip["start"]<=s["end"]<=clip["end"])
            meta = gen_facebook_meta(clip, seg_text, output_lang,
                                      gemini_key, model_id, fallback_ids, logs)
            j_path = path.replace(".mp4","_meta.json")
            Path(j_path).write_text(json.dumps(meta,ensure_ascii=False,indent=2))
            meta_lines.append(f"── Clip {i+1} ──\n📌 {meta.get('title','')}\n"
                               f"📝 {meta.get('caption','')[:100]}…\n"
                               f"🏷️ {' '.join(meta.get('hashtags',[])[:5])}")

        progress(1.0, desc="Done!")
        status = f"✅ {len(out_paths)}/{len(clips)} clips ready\n\n" + "\n\n".join(meta_lines)
        yield "\n".join(logs[-10:]), out_paths, status

    except Exception as e:
        import traceback
        tb = traceback.format_exc()
        log(f"❌ Pipeline error: {e}", logs)
        yield "\n".join(logs[-10:]), [], f"❌ Error: {e}\n{tb[-500:]}"

def run_dubbing(video_file, target_lang_choice, tts_model,
                gemini_key, primary_model, fallback_model,
                progress=gr.Progress()):
    logs = []
    if not video_file:
        yield "❌ No video file", None
        return
    if not gemini_key.strip():
        yield "❌ Gemini API key required (used for translation)", None
        return

    target_lang = {"🇬🇧 English":"en","🇫🇷 French":"fr","🇲🇦 Arabic/Darija":"ar",
                   "🇸🇦 Arabic MSA":"ar","🇩🇪 German":"de","🇮🇹 Italian":"it",
                   "🇨🇳 Chinese":"zh"}.get(target_lang_choice,"en")
    model_id = GEMINI_MODELS.get(primary_model,"gemini-2.5-flash")
    fallback_ids = [GEMINI_MODELS.get(fallback_model,"gemini-1.5-flash"),"gemini-1.5-flash"]

    try:
        progress(0.05, desc="Freeing GPU memory…")
        log(f"GPU: {gpu_mem_str()}", logs)
        free_gpu(logs, clear_whisper=True)
        yield "\n".join(logs[-5:]), None

        progress(0.15, desc="Transcribing source…")
        log("Transcribing source for dubbing…", logs)
        tr = transcribe(video_file, "medium", None, logs)
        yield "\n".join(logs[-6:]), None

        progress(0.35, desc=f"Dubbing with {tts_model.split(' ')[0]}… (can take a few minutes)")
        result, used = dub_video(video_file, tr, target_lang, tts_model,
                                  gemini_key, model_id, fallback_ids, logs)
        progress(1.0)
        tag = "⚠️ FALLBACK — Edge TTS, no voice clone" if used == "edge-tts" else f"✅ {used}"
        log(f"Engine used: {tag}", logs)
        yield "\n".join(logs[-10:]), result

    except torch.cuda.OutOfMemoryError as e:
        free_gpu(logs, clear_whisper=True)
        yield (f"❌ GPU out of memory.\n{e}\n\n"
               f"Click '🧹 Free GPU Memory', try a shorter clip, or pick Edge TTS.\n"
               f"{gpu_mem_str()}"), None
    except Exception as e:
        import traceback
        log(f"❌ {e}", logs)
        yield "\n".join(logs[-8:]) + f"\n\n{traceback.format_exc()[-400:]}", None

# ═══════════════════════════════════════════════════════════════
# BUILD GRADIO INTERFACE
# ═══════════════════════════════════════════════════════════════
DESCRIPTION = """
## 🎬 Telfaza Zman — Clip Generator & Dubbing Studio
Generate viral Facebook Reels from long videos with AI. Arabic/Darija optimized.
"""

with gr.Blocks(title="Telfaza Zman", theme=gr.themes.Soft()) as demo:
    gr.Markdown(DESCRIPTION)

    with gr.Tabs():

        # ── TAB 1: CLIP GENERATOR ─────────────────────────────
        with gr.Tab("🎬 Clip Generator"):
            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown("### 📹 Video Input")
                    video_source = gr.Radio(
                        ["Upload file","URL (YouTube/Facebook)"],
                        value="Upload file", label="Source")
                    video_input = gr.Video(label="Upload video", visible=True)
                    url_input = gr.Textbox(label="Video URL",
                        placeholder="https://youtube.com/watch?v=...", visible=False)
                    video_source.change(
                        lambda s: [gr.update(visible=s=="Upload file"),
                                   gr.update(visible=s!="Upload file")],
                        video_source, [video_input, url_input])

                    cookies_file = gr.File(
                        label="YouTube cookies.txt (optional — fixes bot detection)",
                        file_types=[".txt"], type="filepath")

                    gr.Markdown("---")
                    gr.Markdown("### 🤖 AI Models")
                    primary_model = gr.Dropdown(
                        choices=list(GEMINI_MODELS.keys()),
                        value=list(GEMINI_MODELS.keys())[0],
                        label="Primary model")
                    fallback_model = gr.Dropdown(
                        choices=list(GEMINI_MODELS.keys()),
                        value=list(GEMINI_MODELS.keys())[2],
                        label="Fallback model (auto-switches on error)")
                    gemini_key_t1 = gr.Textbox(
                        label="Gemini API Key", type="password",
                        placeholder="AIza…")

                    gr.Markdown("---")
                    gr.Markdown("### 🎙️ Transcription")
                    whisper_model = gr.Dropdown(
                        choices=WHISPER_MODELS, value="large-v3",
                        label="Whisper model size")
                    whisper_lang = gr.Dropdown(
                        choices=list(WHISPER_LANG.keys()),
                        value="🔍 Auto-detect",
                        label="Source language (force or auto-detect)")

                    gr.Markdown("---")
                    gr.Markdown("### 🌍 Output Language")
                    output_lang = gr.Dropdown(
                        choices=list(LANGUAGES.keys()),
                        value=list(LANGUAGES.keys())[0],
                        label="Language for titles / descriptions / hooks")

                with gr.Column(scale=1):
                    gr.Markdown("### ⚙️ Clip Settings")
                    with gr.Row():
                        min_clip_dur = gr.Slider(30,90,value=60,step=5,
                            label="Min clip duration (s)")
                        max_clips = gr.Slider(3,20,value=10,step=1,
                            label="Max clips to find")

                    gr.Markdown("### 🎨 Watermark & Branding")
                    wm_text = gr.Textbox(value="Telfaza Zman",
                        label="Watermark text (leave empty to disable)")
                    wm_pos = gr.Dropdown(choices=WM_POSITIONS,
                        value="bottom-center", label="Position")
                    wm_opacity = gr.Slider(10,100,value=75,step=5,
                        label="Opacity %")
                    wm_logo = gr.File(label="Logo PNG (optional — replaces text)",
                        file_types=["image"], type="filepath")

                    gr.Markdown("### 🎨 Visual Style")
                    neon_color = gr.Dropdown(
                        choices=["0x00FFFF (Cyan)","0xFF00FF (Magenta)",
                                 "0xFFFF00 (Yellow)","0xFF6B00 (Orange)",
                                 "0x00FF00 (Green)"],
                        value="0x00FFFF (Cyan)",
                        label="Split-screen neon divider color")
                    sub_size = gr.Slider(10,28,value=16,step=1,
                        label="Subtitle font size")

                    with gr.Row():
                        gen_btn = gr.Button("🚀 Generate Clips", variant="primary", size="lg")
                        cancel_btn = gr.Button("⏹️ Cancel", size="lg")
                    free_gpu_btn1 = gr.Button("🧹 Free GPU Memory / Reset Models")

            # Results
            with gr.Row():
                log_out = gr.Textbox(label="📋 Live Logs", lines=12,
                    interactive=False, max_lines=20)
                status_out = gr.Textbox(label="📊 Status & Metadata",
                    lines=12, interactive=False)

            gallery = gr.Gallery(label="🎬 Generated Clips",
                                  columns=3, height="auto",
                                  object_fit="contain")

            def parse_neon(nc):
                return nc.split(" ")[0]

            def run_pipeline_ui(vi,ui,vs,gk,pm,fm,wm,wl,ol,mcd,mc,wt,wp,wo,wlo,nc,ss,cf,
                                 progress=gr.Progress()):
                yield from run_pipeline(vi,ui,vs,gk,pm,fm,wm,wl,ol,mcd,mc,wt,wp,wo,wlo,
                                         parse_neon(nc),ss,cf,progress=progress)

            gen_event = gen_btn.click(
                fn=run_pipeline_ui,
                inputs=[video_input,url_input,video_source,
                        gemini_key_t1,primary_model,fallback_model,
                        whisper_model,whisper_lang,output_lang,
                        min_clip_dur,max_clips,
                        wm_text,wm_pos,wm_opacity,wm_logo,
                        neon_color,sub_size,cookies_file],
                outputs=[log_out, gallery, status_out]
            )
            cancel_btn.click(fn=None, inputs=None, outputs=None, cancels=[gen_event])
            free_gpu_btn1.click(fn=free_gpu_ui, outputs=log_out)

        # ── TAB 2: DUBBING STUDIO ─────────────────────────────
        with gr.Tab("🎙️ Dubbing Studio"):
            gr.Markdown("""
### Voice-Cloning Dubbing Pipeline
Translates audio to another language while **preserving the original speaker's voice**.
Transcript is grouped into full sentences (not raw Whisper fragments) and
time-aligned gently — this is what prevents the "robotic" sound of
per-fragment TTS.

| Model | Quality | Arabic? | Notes |
|---|---|---|---|
| **Chatterbox Multilingual** ⭐ | ⭐⭐⭐⭐⭐ | ✅ Native | Recommended. 23 languages, cross-language voice transfer (e.g. ES voice → AR speech) |
| **XTTS-v2** | ⭐⭐⭐⭐ | ✅ | Good fallback, ~6s reference |
| **Chatterbox (English)** | ⭐⭐⭐⭐⭐ | ❌ | Best if target language is English only |
| **Edge TTS** | ⭐⭐⭐ | generic | No GPU, **no voice cloning** — last resort only |

> First run of any model downloads weights (1-2GB) — cached after that.
> If a job stalls or you see a GPU memory error, click **🧹 Free GPU Memory**
> below instead of restarting the whole runtime.
            """)
            with gr.Row():
                with gr.Column():
                    dub_video_in = gr.Video(label="Video to dub")
                    dub_target = gr.Dropdown(
                        choices=["🇬🇧 English","🇫🇷 French","🇲🇦 Arabic/Darija",
                                 "🇸🇦 Arabic MSA","🇩🇪 German","🇮🇹 Italian","🇨🇳 Chinese"],
                        value="🇲🇦 Arabic/Darija",
                        label="Target language for dubbing")
                    dub_model = gr.Dropdown(
                        choices=["Chatterbox Multilingual (BEST — Arabic native)",
                                 "XTTS-v2 (multilingual, Arabic support)",
                                 "Chatterbox (English only, best EN quality)",
                                 "Edge TTS (free, no GPU, no voice clone)"],
                        value="Chatterbox Multilingual (BEST — Arabic native)",
                        label="TTS / Voice cloning model")
                    gr.Markdown("*Uses the same Gemini API key entered in the Clip Generator tab.*")
                    pm_t2 = gr.Dropdown(choices=list(GEMINI_MODELS.keys()),
                        value=list(GEMINI_MODELS.keys())[0], label="Gemini model")
                    fm_t2 = gr.Dropdown(choices=list(GEMINI_MODELS.keys()),
                        value=list(GEMINI_MODELS.keys())[2], label="Fallback")
                    with gr.Row():
                        dub_btn = gr.Button("🎙️ Generate Dubbed Video",
                                            variant="primary", size="lg")
                        dub_cancel_btn = gr.Button("⏹️ Cancel", size="lg")
                    free_gpu_btn2 = gr.Button("🧹 Free GPU Memory")

                with gr.Column():
                    dub_log = gr.Textbox(label="Dubbing Log", lines=14, interactive=False)
                    dub_result = gr.Video(label="Dubbed Video")

            dub_event = dub_btn.click(
                fn=run_dubbing,
                inputs=[dub_video_in,dub_target,dub_model,
                        gemini_key_t1,pm_t2,fm_t2],
                outputs=[dub_log, dub_result])
            dub_cancel_btn.click(fn=None, inputs=None, outputs=None, cancels=[dub_event])
            free_gpu_btn2.click(fn=free_gpu_ui, outputs=dub_log)

        # ── TAB 3: SETUP & TIPS ───────────────────────────────
        with gr.Tab("⚙️ Setup & Tips"):
            gr.Markdown("""
## Quick Setup Guide

### 1. API Keys
- **Gemini** (required): [aistudio.google.com/app/apikey](https://aistudio.google.com/app/apikey) — free tier, 15 req/min
- **Gemini API is the only required key.** Everything else is optional.

### 2. YouTube Bot Detection Fix
YouTube blocks Colab and RunPod IPs. Three workarounds (best first):
1. **Cookies export**: Install Chrome extension "Get cookies.txt LOCALLY" → log in to YouTube → export → upload in the Clip Generator tab
2. **Upload manually**: Download on your laptop → upload via the file picker
3. **Facebook URLs**: Paste Facebook video URLs directly — they work fine without cookies

### 3. Speed by Model (RTX 4090 / Colab T4)
| Task | RTX 4090 | Colab T4 |
|---|---|---|
| Whisper large-v3 (60s video) | ~20s | ~90s |
| Whisper large-v3 (30min video) | ~6min | ~25min |
| Gemini clip detection | ~5-15s | same |
| FFmpeg clip + subtitles | ~30s/clip | ~45s/clip |
| XTTS-v2 dubbing (1min clip) | ~2min | ~5min |

### 4. Dubbing Tips
- XTTS-v2 first run downloads ~2GB model (cached after that)
- For Arabic dubbing: use XTTS-v2 with target "Arabic/Darija"
- For English dubbing from Arabic: use Chatterbox for best quality
- Edge TTS is always available as fallback (no GPU, no voice clone)

### 5. Save to Google Drive
Clips auto-save to `MyDrive/TelfazaZman/clips/` when Drive is mounted.
To mount Drive, run this in a separate Colab cell:
```python
from google.colab import drive
drive.mount('/content/drive')
```

### 6. Recommended pyvideotrans (standalone app)
For a full desktop dubbing solution, try [pyvideotrans](https://github.com/jianchang512/pyvideotrans):
- Windows .exe installer, no Python needed
- Supports F5-TTS, CosyVoice, GPT-SoVITS, Chatterbox for voice cloning
- Handles timing alignment automatically
- Supports 40+ languages
            """)

# ─── Launch ───────────────────────────────────────────────────
# max_size=1 + default_concurrency_limit=1: only one job runs at a time,
# so the Cancel button always targets a real running job (previously,
# a crashed worker could leave "queued" jobs with nothing to cancel).
if __name__ == "__main__":
    demo.queue(max_size=1, default_concurrency_limit=1).launch(
        share=True,           # generates public URL (needed in Colab)
        server_port=7860,
        show_error=True,
    )
else:
    # Called via exec() inside a Colab cell
    demo.queue(max_size=1, default_concurrency_limit=1).launch(share=True, show_error=True)

/tmp/ipykernel_12502/2763318120.py:975: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Telfaza Zman", theme=gr.themes.Soft()) as demo:
/tmp/ipykernel_12502/2763318120.py:1093: DeprecationWarning: The 'show_api' parameter in event listeners will be removed in Gradio 6.0. You will need to use the 'api_visibility' parameter instead. To replicate show_api=False, in Gradio 6.0, use api_visibility='undocumented'.
  cancel_btn.click(fn=None, inputs=None, outputs=None, cancels=[gen_event])
/tmp/ipykernel_12502/2763318120.py:1151: DeprecationWarning: The 'show_api' parameter in event listeners will be removed in Gradio 6.0. You will need to use the 'api_visibility' parameter instead. To replicate show_api=False, in Gradio 6.0, use api_visibility='undocumented'.
  dub_cancel_btn.click(fn=None, inputs=None, outputs=None, cancels=[dub_event])


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://71e734a23d4ba675c2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
